# AI Business Advisor Workflow

An AI Business Advisor that:

1. Accepts an industry as input.
2. Generates a business idea.
3. Analyzes the strengths and weaknesses.
4. Formats the results as a final report.


## 0. Import the necessary libs

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough
from pydantic import BaseModel, Field

## 1. Instantiate Chat Model with your API Key

Set `OPENAI_API_KEY` in your environment before running the model cell. LangChain's `ChatOpenAI` reads this variable automatically; avoid putting secrets directly in a notebook.

```powershell
$env:OPENAI_API_KEY = "your-api-key"
```

In [ ]:
# Set OPENAI_API_KEY in your environment before running this cell.
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
)

## 2. Save Logs

In the end of each chain, parse the output and save the logs

In [ ]:
logs = []

In [ ]:
parser = StrOutputParser()

In [ ]:
parse_and_log_output_chain = RunnableParallel(
    output=parser, 
    log=RunnableLambda(lambda x: logs.append(x))
)

## 3. Idea Generation

Craft a prompt to generate a business idea for the given industry. 

In [ ]:
idea_prompt = PromptTemplate(
    input_variables=["industry"],
    template=(
        "You are a practical business ideation advisor. Propose one specific, "
        "realistic business idea for the {industry} industry. Describe the target "
        "customer, the problem solved, the product or service, how it earns revenue, "
        "and one assumption that should be tested. Keep the idea concise and avoid "
        "unsupported market-size claims."
    ),
)

In [ ]:
idea_chain = idea_prompt | llm | parse_and_log_output_chain

In [ ]:
idea_result = idea_chain.invoke({"industry": "agriculture"})

In [ ]:
idea_result["output"]

In [ ]:
logs

## 4. Idea Analysis

Craft a prompt to analyze the generated idea's strengths and weaknesses

In [ ]:
analysis_prompt = PromptTemplate(
    input_variables=["idea"],
    template=(
        "Evaluate this business idea as a cautious advisor.\n\n"
        "Business idea:\n{idea}\n\n"
        "Explain the strongest advantages and the most important weaknesses. "
        "Consider customer demand, operations, competition, costs, and key risks. "
        "Be specific, distinguish assumptions from facts, and do not invent data."
    ),
)

In [ ]:
analysis_chain = analysis_prompt | llm | parse_and_log_output_chain

In [ ]:
analysis_result = analysis_chain.invoke({"idea": idea_result["output"]})

In [ ]:
analysis_result["output"]

In [ ]:
logs

## 5. Report Generation

Craft a prompt to structure the information into a business report.

In [ ]:
report_prompt = PromptTemplate(
    input_variables=["analysis"],
    template=(
        "Turn the advisor's analysis below into a concise, decision-ready business "
        "report. Return an executive_summary, a list of actionable strengths, a "
        "list of material weaknesses or risks, and practical recommendations for "
        "the next validation steps. Base every point only on the analysis; note "
        "uncertainty instead of inventing facts.\n\n"
        "Advisor analysis:\n{analysis}"
    ),
)

In [ ]:
class AnalysisReport(BaseModel):
    """Structured assessment of a business idea."""

    executive_summary: str = Field(description="Brief decision-ready summary")
    strengths: list[str] = Field(description="Specific advantages of the idea")
    weaknesses: list[str] = Field(description="Material weaknesses and risks")
    recommendations: list[str] = Field(description="Practical next steps to validate the idea")

In [ ]:
report_chain = report_prompt | llm.with_structured_output(AnalysisReport)

In [ ]:
report_result = report_chain.invoke({"analysis": analysis_result["output"]})

In [ ]:
report_result

## 6. End to End Chain

In [ ]:
e2e_chain = (
    RunnableParallel(industry=RunnablePassthrough())
    | idea_chain
    | RunnablePassthrough.assign(
        idea=RunnableLambda(lambda result: result["output"])
    )
    | analysis_chain
    | RunnablePassthrough.assign(
        analysis=RunnableLambda(lambda result: result["output"])
    )
    | report_chain
)

In [ ]:
try:
    e2e_chain.get_graph().print_ascii()
except ImportError:
    print("Install grandalf to display the workflow graph: pip install grandalf")

In [ ]:
# Change the industry if you want
e2e_result = e2e_chain.invoke("agro")

In [ ]:
e2e_result

In [ ]:
e2e_result.strengths

In [ ]:
e2e_result.weaknesses

In [ ]:
logs

## 7. Extension

- Added session memory so a follow-up can refer to an earlier turn.
- Use `RunnableBranch` to route ideas to different analyses by business risk.
- Added a fact-checking or market-research step, and clearly label assumptions versus verified evidence.
- Extended the structured report with a scorecard or a short list of experiments.

The cells below demonstrate in-memory chat history. A session ID keeps each conversation separate; the history lasts only for the current Python process.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

history_store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    return history_store.setdefault(session_id, InMemoryChatMessageHistory())

In [ ]:
memory_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a practical business advisor. Use the conversation history when answering follow-up questions, and be clear about assumptions."),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}"),
    ]
)

memory_chain = RunnableWithMessageHistory(
    memory_prompt | llm | parser,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)

In [ ]:
memory_config = {"configurable": {"session_id": "business-advisor-demo"}}

first_turn = memory_chain.invoke(
    {"question": "Suggest a business idea for small-scale agriculture."},
    config=memory_config,
)
second_turn = memory_chain.invoke(
    {"question": "What is one low-cost way to test demand for that idea?"},
    config=memory_config,
)

print("First turn:\n", first_turn)
print("\nFollow-up using session history:\n", second_turn)
print("\nMessages stored:", len(history_store["business-advisor-demo"].messages))